In [2]:
import torch
import numpy as np

# для colab способ 1
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/model.py', '/content/model.py')

# для colab способ 2
# from google.colab import drive
# drive.mount('/content/drive')
# sys.path.append('/content/drive/MyDrive')

from model import SASRecModel, negative_sampling_loss

Mounted at /content/drive


In [3]:
def ndcg_at_k(rel, pred, k=10):
    # метрика NDCG@k
    ndcg = 0.0
    if rel in pred[:k]:
        pred_list = list(pred[:k])
        score = pred_list.index(rel) + 1
        ndcg = 1.0 / np.log2(score + 1)
        return ndcg
    return ndcg


def recall_at_k(rel, pred, k=10):
    # метрика Recall@k
    recall = 1.0 if rel in pred[:k] else 0.0
    return recall

In [7]:
# Тест 1 — размерность выхода модели (метод белого ящика)

model = SASRecModel(cnt_item=100, max_seq_len=10)
x = torch.randint(1, 100, (4, 10))
out = model(x)
print(out.shape)
assert out.shape == (4, 10, 101), f'Ожидалось (4, 10, 101), получили {out.shape}'
print('Тест пройден')

torch.Size([4, 10, 101])
Тест пройден


In [12]:
# Тест 2 — Causal mask (метод белого ящика)

seq_len = 5
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
print(mask)
assert mask[0, 1] == True, 'Позиция 0 не должна видеть позицию 1 (будущее)'
assert mask[1, 0] == False, 'Позиция 1 должна видеть позицию 0 (прошлое)'
assert mask[1, 2] == True, 'Позиция 1 не должна видеть позицию 2 (будущее)'
print('Тест пройден')

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
Тест пройден


In [16]:
# Тест 3 — predict_next (модульное тестирование)

model = SASRecModel(cnt_item=100, max_seq_len=10)
history = [5, 10, 15, 20]
items, scores = model.predict_next(history, top_k=5)
print(items)
print(scores)
assert len(items) == 5, f'Ожидалось 5, получили {len(items)}'
assert len(scores) == 5, f'Ожидалось 5, получили {len(scores)}'
print('Тест пройден')

[42  2 99 37 53]
[2.473951  1.9275302 1.8694543 1.8558079 1.7578354]
Тест пройден
